# A notebook to construct a custom training/validation and test datasets from SOCOFing Real + Altered-Easy sanitized images 
# Train dataset : consider unaltered images from 'Real' directory
#                 consider augmented 'Zcut' (cut) , 'Obl' (blur) , 'CR' (drop) images from 'Altered-Easy' directory
# Validation dataset : consider unaltered images from 'Real' directory
#                      consider augmented 'Zcut' (cut) images from 'Altered-Easy' directory (a cut is usual for a fingrrprint)
# Maintain 80 / 20 ratio betwwen train / validation datasets

In [ ]:
# Import librairies
import os
import numpy as np
import pandas as pd

from PIL import Image
import matplotlib.pyplot as plt

## Explore the training dataset

In [ ]:
# Distribution graphs (histogram/bar graph) of column data
def plotPerColumnDistribution(df, nGraphShown, nGraphPerRow):
    nunique = df.nunique()
    df = df[[col for col in df if nunique[col] > 1 and nunique[col] < 50]] # For displaying purposes, pick columns that have between 1 and 50 unique values
    nRow, nCol = df.shape
    columnNames = list(df)
    nGraphRow = int((nCol + nGraphPerRow - 1) / nGraphPerRow)
    plt.figure(num = None, figsize = (6 * nGraphPerRow, 8 * nGraphRow), dpi = 80, facecolor = 'w', edgecolor = 'k')
    for i in range(min(nCol, nGraphShown)):
        plt.subplot(nGraphRow, nGraphPerRow, i + 1)
        columnDf = df.iloc[:, i]
        if (not np.issubdtype(type(columnDf.iloc[0]), np.number)):
            valueCounts = columnDf.value_counts()
            valueCounts.plot.bar()
        else:
            columnDf.hist()
        plt.ylabel('counts')
        plt.xticks(rotation = 90)
        plt.title(f'{columnNames[i]} (column {i})')
    plt.tight_layout(pad = 1.0, w_pad = 1.0, h_pad = 1.0)
    plt.show()

In [ ]:
# Constants for training/validation dataset

# Sub-directories to traverse
# SCAN_TRAINING_SUBDIRECTORIES = ['Real', 'Altered-Easy', 'Altered-Medium', 'Altered-Hard']

# Define top-level directory and output CSV path
DATASET_ROOT_DIR = "/Users/laurent/Projects/projet-fingerprint-validator/dataset"
DATASET_ROOT_NAME = "SOCOFing"

DATASET_TRAINING_DIR = DATASET_ROOT_DIR + "/" + DATASET_ROOT_NAME + "_Training"
DATASET_TRAINING_CSV_PATH = DATASET_TRAINING_DIR + "_with_labels.csv"

DATASET_CUSTOM_DIR = DATASET_ROOT_DIR + "/" + DATASET_ROOT_NAME + "_Custom"
OUTPUT_DATASET_CUSTOM_CSV_PATH = DATASET_CUSTOM_DIR + "_with_features.csv"

In [ ]:
# load csv
df = pd.read_csv(DATASET_TRAINING_CSV_PATH)

print(f"\n📊 Fingerprints dataset basic stats :")
df.info()

In [ ]:
# Display the statistical summary of the dataset
df.describe()

In [ ]:
# show null values
df.isnull().sum()

In [ ]:
# Display the distribution of finger (label)
print(df["Finger"].value_counts())

In [ ]:
# show distribution of columns
plotPerColumnDistribution(df, 10, 5)

In [ ]:
# Display of number of thumbs, index, middle, ring, and little fingers
if len(df):
    print(f"Percentage of Thumbs: {df['Finger'].value_counts().get('thumb', 0) / len(df) * 100:.2f}%")
    print(f"Percentage of Index fingers: {df['Finger'].value_counts().get('index', 0) / len(df) * 100:.2f}%")
    print(f"Percentage of Middle fingers: {df['Finger'].value_counts().get('middle', 0) / len(df) * 100:.2f}%")
    print(f"Percentage of Ring fingers: {df['Finger'].value_counts().get('ring', 0) / len(df) * 100:.2f}%")
    print(f"Percentage of Little fingers: {df['Finger'].value_counts().get('little', 0) / len(df) * 100:.2f}%")

## Create a new frame with features

In [ ]:
# List to store file information and images features in the original dataset
data = []

# Traverse the input directory to get all image files
for dirname, _, filenames in os.walk(DATASET_TRAINING_DIR):
    for filename in filenames:

        if filename == ".DS_Store":  # Skip macOS system files
            continue
        
        # Extract the features from the dirname and the filename
        file_path = os.path.join(dirname, filename)
        
        if filename.find("Zcut") != -1:
            feature = "zcut"
        elif filename.find("CR") != -1:
            feature = "cr"
        elif filename.find("Obl") != -1:
            feature = "obl"
        else:
            feature = "unaltered"

        if filename.find("thumb") != -1:
            finger = "thumb"
        elif filename.find("index") != -1:
            finger = "index"
        elif filename.find("middle") != -1:
            finger = "middle"
        elif filename.find("ring") != -1:
            finger = "ring"
        elif filename.find("little") != -1:
            finger = "little"
        else:
            finger = None

        # Extract the image features
        img = Image.open(file_path)
        try:
            img.verify()  # Verify that the file is a valid image
        except:
            print(f"Invalid image file: {file_path}")
            continue
        
        data.append([file_path, feature, finger])

# Create a DataFrame
df = pd.DataFrame(data, columns=["ImagePath", "Feature", "Finger"])

# Add an index column
df.index.name = "ID"

# Save to CSV
df.to_csv(OUTPUT_DATASET_CUSTOM_CSV_PATH)

print(f"Full Organised Dataset CSV file with features saved at: {OUTPUT_DATASET_CUSTOM_CSV_PATH}")

In [ ]:
df.describe()

In [ ]:
# show distribution of columns
plotPerColumnDistribution(df, 10, 5)

In [ ]:
# Display of features
if len(df):
    print(f"Percentage of Real: {df['Feature'].value_counts().get('unaltered', 0) / len(df) * 100:.2f}%")
    print(f"Percentage of Zcut: {df['Feature'].value_counts().get('zcut', 0) / len(df) * 100:.2f}%")
    print(f"Percentage of CR: {df['Feature'].value_counts().get('cr', 0) / len(df) * 100:.2f}%")
    print(f"Percentage of Obl: {df['Feature'].value_counts().get('obl', 0) / len(df) * 100:.2f}%")


In [ ]:
# Display the distribution of features
print(df["Feature"].value_counts())

In [ ]:
# List to store file information and images features in the original dataset
data_thumb = []
data_index = []
data_middle = []
data_ring = []
data_little = []

# Traverse the input directory to get all image files
for dirname, _, filenames in os.walk(DATASET_TRAINING_DIR):
    for filename in filenames:

        if filename == ".DS_Store":  # Skip macOS system files
            continue

        if filename.find('thumb') != -1:
            data = data_thumb
            finger = 'thumb'
        elif filename.find('index') != -1:
            data = data_index
            finger = 'index'
        elif filename.find('middle') != -1:
            data = data_middle
            finger = 'middle'
        elif filename.find('ring') != -1:
            data = data_ring
            finger = 'ring'
        elif filename.find('little') != -1:
            data = data_little
            finger = 'little'
        else:
            continue

        # Extract the features from the dirname and the filename
        file_path = os.path.join(dirname, filename)
        
        if filename.find("Zcut") != -1:
            feature = "zcut"
        elif filename.find("CR") != -1:
            feature = "cr"
        elif filename.find("Obl") != -1:
            feature = "obl"
        else:
            feature = "unaltered"
        
        data.append([file_path, feature, finger])

# Create DataFrames
df_thumb = pd.DataFrame(data_thumb, columns=["ImagePath", "Feature", "Finger"])
df_thumb.index.name = "ID"

df_index = pd.DataFrame(data_index, columns=["ImagePath", "Feature", "Finger"])
df_index.index.name = "ID"

df_middle = pd.DataFrame(data_middle, columns=["ImagePath", "Feature", "Finger"])
df_middle.index.name = "ID"

df_ring = pd.DataFrame(data_ring, columns=["ImagePath", "Feature", "Finger"])
df_ring.index.name = "ID"

df_little = pd.DataFrame(data_little, columns=["ImagePath", "Feature", "Finger"])
df_little.index.name = "ID"


In [ ]:
plotPerColumnDistribution(df_thumb, 10, 5)

In [ ]:
plotPerColumnDistribution(df_index, 10, 5)

In [ ]:
plotPerColumnDistribution(df_middle, 10, 5)

In [ ]:
plotPerColumnDistribution(df_ring, 10, 5)

In [ ]:
plotPerColumnDistribution(df_little, 10, 5)

In [ ]:
def sort_feature(df):
    
    data_unaltered = []
    data_zcut = []
    data_cr = []
    data_obl = []

    for i in range(0, len(df)):
        if df.at[i, 'Feature'] == 'unaltered':
            data = data_unaltered
        elif df.at[i, 'Feature'] == 'zcut':
            data = data_zcut
        elif df.at[i, 'Feature'] == 'cr':
            data = data_cr
        elif df.at[i, 'Feature'] == 'obl':
            data = data_obl
        else:
            continue

        data.append(df.at[i, 'ImagePath'])

    # Create a DataFrame
    df_unaltered = pd.DataFrame(data_unaltered, columns=["ImagePath"])
    df_unaltered.index.name = "ID"

    df_zcut = pd.DataFrame(data_zcut, columns=["ImagePath"])
    df_zcut.index.name = "ID"

    df_cr = pd.DataFrame(data_cr, columns=["ImagePath"])
    df_cr.index.name = "ID"

    df_obl = pd.DataFrame(data_obl, columns=["ImagePath"])
    df_obl.index.name = "ID"

    return df_unaltered, df_zcut, df_cr, df_obl

In [ ]:
df_thumb_unaltered, df_thumb_zcut, df_thumb_cr, df_thumb_obl = sort_feature(df_thumb)
df_index_unaltered, df_index_zcut, df_index_cr, df_index_obl = sort_feature(df_index)
df_middle_unaltered, df_middle_zcut, df_middle_cr, df_middle_obl = sort_feature(df_middle)
df_ring_unaltered, df_ring_zcut, df_ring_cr, df_ring_obl = sort_feature(df_ring)
df_little_unaltered, df_little_zcut, df_little_cr, df_little_obl = sort_feature(df_little)


In [ ]:
TARGET_ROOT_DIR = "/Users/laurent/Projects/projet-fingerprint-validator/dataset/SOCOFing_Custom/"

def random_choose_and_copy_images(imgdf, target_root_dir, feature, finger):

    target_val_dir = target_root_dir + "validation/" + finger
    target_train_dir = target_root_dir + "train/" + finger

    imgtp = np.random.choice(imgdf['ImagePath'], 400, replace=False)
    imgdict = {}

    for i in range(len(imgtp)):
        imgdict[imgtp[i]] = True

        if feature == 'unaltered' or feature == 'zcut':
            # copy image
            source_file_path = imgtp[i]
            filename = os.path.basename(source_file_path)
            target_file_path = target_val_dir + "/" + filename
            img = Image.open(source_file_path)
            img.save(target_file_path)

    for i in range(len(imgdf)):
        if imgdf.at[i, 'ImagePath'] not in imgdict:
            # copy image
            source_file_path = imgdf.at[i, 'ImagePath']
            filename = os.path.basename(source_file_path)
            target_file_path = target_train_dir + "/" + filename
            img = Image.open(source_file_path)
            img.save(target_file_path)

In [ ]:
print(f"Copying thumb images")
random_choose_and_copy_images(df_thumb_unaltered, TARGET_ROOT_DIR, 'unaltered', 'thumb')
random_choose_and_copy_images(df_thumb_zcut, TARGET_ROOT_DIR, 'zcut', 'thumb')
random_choose_and_copy_images(df_thumb_cr, TARGET_ROOT_DIR, 'cr', 'thumb')
random_choose_and_copy_images(df_thumb_obl, TARGET_ROOT_DIR, 'obl', 'thumb')

print(f"Copying index images")
random_choose_and_copy_images(df_index_unaltered, TARGET_ROOT_DIR, 'unaltered', 'index')
random_choose_and_copy_images(df_index_zcut, TARGET_ROOT_DIR, 'zcut', 'index')
random_choose_and_copy_images(df_index_cr, TARGET_ROOT_DIR, 'cr', 'index')
random_choose_and_copy_images(df_index_obl, TARGET_ROOT_DIR, 'obl', 'index')

print(f"Copying middle images")
random_choose_and_copy_images(df_middle_unaltered, TARGET_ROOT_DIR, 'unaltered', 'middle')
random_choose_and_copy_images(df_middle_zcut, TARGET_ROOT_DIR, 'zcut', 'middle')
random_choose_and_copy_images(df_middle_cr, TARGET_ROOT_DIR, 'cr', 'middle')
random_choose_and_copy_images(df_middle_obl, TARGET_ROOT_DIR, 'obl', 'middle')

print(f"Copying ring images")
random_choose_and_copy_images(df_ring_unaltered, TARGET_ROOT_DIR, 'unaltered', 'ring')
random_choose_and_copy_images(df_ring_zcut, TARGET_ROOT_DIR, 'zcut', 'ring')
random_choose_and_copy_images(df_ring_cr, TARGET_ROOT_DIR, 'cr', 'ring')
random_choose_and_copy_images(df_ring_obl, TARGET_ROOT_DIR, 'obl', 'ring')

print(f"Copying little images")
random_choose_and_copy_images(df_little_unaltered, TARGET_ROOT_DIR, 'unaltered', 'little')
random_choose_and_copy_images(df_little_zcut, TARGET_ROOT_DIR, 'zcut', 'little')
random_choose_and_copy_images(df_little_cr, TARGET_ROOT_DIR, 'cr', 'little')
random_choose_and_copy_images(df_little_obl, TARGET_ROOT_DIR, 'obl', 'little')

In [ ]:
TARGET_ROOT_DIR = "/Users/laurent/Projects/projet-fingerprint-validator/dataset/SOCOFing_Custom/"

def random_choose_unaltered_and_copy_associated_images(imgdf, target_root_dir, finger):

    target_val_dir = target_root_dir + "validation/" + finger
    target_train_dir = target_root_dir + "train/" + finger

    imgtp = np.random.choice(imgdf['ImagePath'], 400, replace=False)
    imgdict = {}

    for i in range(len(imgtp)):
        imgdict[imgtp[i]] = True

        # copy image
        source_file_path = imgtp[i]
        filename = os.path.basename(source_file_path)
        target_file_path = target_val_dir + "/" + filename

        img = Image.open(source_file_path)
        img.save(target_file_path)

    for i in range(len(imgdf)):
        if imgdf.at[i, 'ImagePath'] not in imgdict:
            # copy image
            source_file_path = imgdf.at[i, 'ImagePath']
            filename = os.path.basename(source_file_path)
            target_file_path = target_train_dir + "/" + filename
            img = Image.open(source_file_path)
            img.save(target_file_path)

            # copy corresponding altered images
            dirname = os.path.dirname(source_file_path)
            basename = filename.replace(".jpeg", "")

            # Zcut
            source_file_path = os.path.join(dirname, basename)
            source_file_path = source_file_path + "_Zcut.jpeg"
            filename = os.path.basename(source_file_path)
            target_file_path = target_train_dir + "/" + filename
            try:
                img = Image.open(source_file_path)
                img.save(target_file_path)
            except:
                print(f"file does not exsits: {os.path.basename(source_file_path)}")

            # CR
            source_file_path = os.path.join(dirname, basename)
            source_file_path = source_file_path + "_CR.jpeg"
            filename = os.path.basename(source_file_path)
            target_file_path = target_train_dir + "/" + filename            
            try:
                img = Image.open(source_file_path)
                img.save(target_file_path)
            except:
                print(f"file does not exsits: {os.path.basename(source_file_path)}")

            # Obl
            source_file_path = os.path.join(dirname, basename)
            source_file_path = source_file_path + "_Obl.jpeg"
            filename = os.path.basename(source_file_path)
            target_file_path = target_train_dir + "/" + filename
            try:
                img = Image.open(source_file_path)
                img.save(target_file_path)
            except:
                print(f"file does not exsits: {os.path.basename(source_file_path)}")

In [ ]:
#print(f"Copying thumb images")
#random_choose_unaltered_and_copy_associated_images(df_thumb_unaltered, TARGET_ROOT_DIR, 'thumb')

#print(f"Copying index images")
#random_choose_unaltered_and_copy_associated_images(df_index_unaltered, TARGET_ROOT_DIR, 'index')

#print(f"Copying middle images")
#random_choose_unaltered_and_copy_associated_images(df_middle_unaltered, TARGET_ROOT_DIR, 'middle')

#print(f"Copying ring images")
#random_choose_unaltered_and_copy_associated_images(df_ring_unaltered, TARGET_ROOT_DIR, 'ring')

#print(f"Copying little images")
#random_choose_unaltered_and_copy_associated_images(df_little_unaltered, TARGET_ROOT_DIR, 'little')

In [ ]:
TARGET_ROOT_DIR = "/Users/laurent/Projects/projet-fingerprint-validator/dataset/SOCOFing_Custom/"

def random_choose_unaltered_and_random_copy_associated_images(imgdf, copy_val, target_root_dir, finger):

    target_val_dir = target_root_dir + "validation/" + finger
    target_train_dir = target_root_dir + "train/" + finger

    if copy_val:
        imgtp = np.random.choice(imgdf['ImagePath'], 400, replace=False)
    else:
        imgtp = np.random.choice(imgdf['ImagePath'], 300, replace=False)

    imgdict = {}

    randomdata = []

    for i in range(len(imgtp)):
        imgdict[imgtp[i]] = True

        if copy_val == True:
            # copy image
            source_file_path = imgtp[i]
            filename = os.path.basename(source_file_path)
            target_file_path = target_val_dir + "/" + filename
            img = Image.open(source_file_path)
            img.save(target_file_path)

    for i in range(len(imgdf)):
        if (copy_val and imgdf.at[i, 'ImagePath'] not in imgdict) or (not copy_val and imgdf.at[i, 'ImagePath'] in imgdict):
            source_orig_file_path = imgdf.at[i, 'ImagePath']
            filename = os.path.basename(source_orig_file_path)
            target_orig_file_path = target_train_dir + "/" + filename

            if copy_val == True:
                # copy image
                img = Image.open(source_orig_file_path)
                img.save(target_orig_file_path)

            # find corresponding altered images
            dirname = os.path.dirname(source_orig_file_path)
            basename = filename.replace(".jpeg", "")

            # Zcut
            source_zcut_file_path = os.path.join(dirname, basename)
            source_zcut_file_path = source_zcut_file_path + "_Zcut.jpeg"
            filename = os.path.basename(source_zcut_file_path)
            target_zcut_file_path = target_train_dir + "/" + filename
            try:
                img = Image.open(source_zcut_file_path)
                if copy_val == False:
                    img.save(target_zcut_file_path)
            except:
                target_zcut_file_path = None

            # CR
            source_cr_file_path = os.path.join(dirname, basename)
            source_cr_file_path = source_cr_file_path + "_CR.jpeg"
            filename = os.path.basename(source_cr_file_path)
            target_cr_file_path = target_train_dir + "/" + filename            
            try:
                img = Image.open(source_cr_file_path)
                if copy_val == False:
                    img.save(target_cr_file_path)
            except:
                target_cr_file_path = None

            # Obl
            source_obl_file_path = os.path.join(dirname, basename)
            source_obl_file_path = source_obl_file_path + "_Obl.jpeg"
            filename = os.path.basename(source_obl_file_path)
            target_obl_file_path = target_train_dir + "/" + filename
            try:
                img = Image.open(source_obl_file_path)
                if copy_val == False:
                    img.save(target_obl_file_path)
            except:
                target_obl_file_path = None
            
            randomdata.append([source_orig_file_path, target_orig_file_path,
                                source_zcut_file_path, target_zcut_file_path,
                                source_cr_file_path, target_cr_file_path,
                                source_obl_file_path, target_obl_file_path])
            
    # Create a DataFrame
    randomdf = pd.DataFrame(randomdata, columns=["ImagePath", "TargetOrigPath", 
                                                 "SourceZCutPath", "TargetZcutPath", 
                                                 "SourceCRPath", "TargetCRPath",
                                                 "SourceOblPath", "TargetOblPath"])

    # Add an index column
    randomdf.index.name = "ID"
    
    return randomdf

In [ ]:
#print(f"Copying thumb images")
#df_thumb_random = random_choose_unaltered_and_random_copy_associated_images(df_thumb_unaltered, True, TARGET_ROOT_DIR, 'thumb')
#df_thumb_random_final = random_choose_unaltered_and_random_copy_associated_images(df_thumb_random, False, TARGET_ROOT_DIR, 'thumb')

#print(f"Copying index images")
#df_index_random = random_choose_unaltered_and_random_copy_associated_images(df_index_unaltered, True, TARGET_ROOT_DIR, 'index')
#df_index_random_final = random_choose_unaltered_and_random_copy_associated_images(df_index_random, False, TARGET_ROOT_DIR, 'index')

#print(f"Copying middle images")
#df_middle_random = random_choose_unaltered_and_random_copy_associated_images(df_middle_unaltered, True, TARGET_ROOT_DIR, 'middle')
#df_middle_random_final = random_choose_unaltered_and_random_copy_associated_images(df_middle_random, False, TARGET_ROOT_DIR, 'middle')

#print(f"Copying ring images")
#df_ring_random = random_choose_unaltered_and_random_copy_associated_images(df_ring_unaltered, True, TARGET_ROOT_DIR, 'ring')
#df_ring_random_final = random_choose_unaltered_and_random_copy_associated_images(df_ring_random, False, TARGET_ROOT_DIR, 'ring')

#print(f"Copying little images")
#df_little_random = random_choose_unaltered_and_random_copy_associated_images(df_little_unaltered, True, TARGET_ROOT_DIR, 'little')
#df_little_random_final = random_choose_unaltered_and_random_copy_associated_images(df_little_random, False, TARGET_ROOT_DIR, 'little')